# Practical 4 — From the Gutenberg–Richter law to aftershock forecasts

By Sebastian Nunez Jara

After a large earthquake, people want to know whether more strong shaking is coming. Several agencies therefore publish **aftershock forecasts**; the Japan Meteorological Agency (JMA), for example, has done so since 1998. Most of these forecasts still use simple statistical models from the 1980s, which "remain the best available models" (Hardebeck et al., 2024). In this practical we build such a forecast ourselves and test how good it is, using earthquake sequences from Japan.

**Plan (90 min)**

| part | content | time |
|---|---|---|
| 0 | The data: 62 earthquake sequences from the JMA catalog | 5 min |
| 1 | The Gutenberg–Richter law: magnitude of completeness $M_c$, b-value and a-value | 35 min |
| 2 | Can b-values help us forecast? (Noto Peninsula, New Year's Eve 2023) | 20 min |
| 3 | Summary | 5 min |

**Reading**

* Hardebeck, J. L., Llenos, A. L., Michael, A. J., Page, M. T., Schneider, M., & van der Elst, N. J. (2024). Aftershock forecasting. *Annual Review of Earth and Planetary Sciences*, 52, 61–84. https://doi.org/10.1146/annurev-earth-040522-102129
* Mirwald, A., Mizrahi, L., Enescu, B., & Wiemer, S. (2026). b-Values of large earthquake sequences depend on their mainshock location. *Geophysical Research Letters*, 53, e2025GL121450. https://doi.org/10.1029/2025GL121450

The notebook runs in the course environment `applied-seismology` (see `README.md` in the course folder), which contains the Python package **SeismoStats** together with numpy, scipy, pandas and matplotlib.

## Prepare the workspace

The first cell below checks that the notebook runs in the course environment.

The earthquake data are downloaded automatically from the [GitHub Release](https://github.com/sebmagia/applied_seismology_rwth/releases/tag/forecasting-practical-v1) of this practical (one file of 8 MB), into the folder `forecasting_practical_data/` next to this notebook.

In [ ]:
# Check that this notebook runs in the course environment (see README.md).
import sys
from importlib.util import find_spec

missing = [pkg for pkg in ('numpy', 'scipy', 'pandas', 'matplotlib', 'seismostats', 'ipympl') if find_spec(pkg) is None]
if missing:
    raise ModuleNotFoundError(
        f"Not installed for this kernel ({sys.executable}): {', '.join(missing)}. "
        "Start Jupyter from the course environment, or select its kernel (see README.md).")
print("Course environment OK:", sys.prefix)

In [ ]:
# Record the versions used in this run.
import sys
import numpy, scipy, pandas, matplotlib, seismostats
print("python", sys.version.split()[0])
for m in (numpy, scipy, pandas, matplotlib, seismostats):
    print(m.__name__, m.__version__)

### Import the packages

In [ ]:
# Interactive figures (zoom, pan) with ipympl, for all the figures of this notebook.
%matplotlib widget

from pathlib import Path
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.max_open_warning"] = 200   # interactive figures stay open: closing one would remove it from the notebook
from scipy import stats, optimize

from seismostats import Catalog
from seismostats.analysis import (ClassicBValueEstimator, BPositiveBValueEstimator,
                                  estimate_mc_maxc, estimate_mc_ks, estimate_a)
from seismostats.plots import plot_cum_fmd, plot_fmd, plot_mc_vs_b
from seismostats.utils import set_option

set_option("warnings", False)   # silence SeismoStats' warnings (we estimate many b-values below)
np.random.seed(42)              # the KS test uses random simulations: make the results reproducible

DELTA_M = 0.1    # magnitudes in the JMA catalog are given in steps (bins) of 0.1

### Download the data

In [ ]:
RELEASE_BASE = (
    "https://github.com/sebmagia/applied_seismology_rwth/"
    "releases/download/forecasting-practical-v1"
)

DATA_ROOT = Path.cwd() / "forecasting_practical_data"   # next to this notebook
DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR = DATA_ROOT / "data"

# Download and extract the archive only if the data folder is not there yet.
if DATA_DIR.exists():
    print("data/ already available")
else:
    zip_path = DATA_ROOT / "data.zip"
    print(f"Downloading {RELEASE_BASE}/data.zip ...")
    urllib.request.urlretrieve(f"{RELEASE_BASE}/data.zip", zip_path)
    print(f"Extracting {zip_path.name} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_ROOT)
    zip_path.unlink(missing_ok=True)

print("Data folder:", DATA_DIR)
for f in sorted(DATA_DIR.iterdir()):
    print(f"  {f.name}  ({f.stat().st_size / 1e6:.1f} MB)")

## Part 0 — The data: 62 earthquake sequences from the JMA catalog

The JMA catalog is one of the densest earthquake catalogs in the world: about 1.8 million events with M ≥ 0.7 between 2000 and 2025 on and around the Japanese islands. We use the **62 earthquake sequences** that Mirwald et al. (2026) extracted from it and published with their code:

* Every M ≥ 6 earthquake (depth ≤ 150 km, ≤ 40 km from the coast) is a *mainshock*. With its rupture length $L$ (Wells & Coppersmith, 1994: $\log_{10} L = -3.22 + 0.69\,M$, in km), the sequence contains all events within a distance of $2L$, from **10 years before** to **100 days after** the mainshock.
* Events of the **first day** after the mainshock are not included, because the catalog is very incomplete right after a large earthquake (more on this in Section 1.6).
* Events that belong to *other* M ≥ 6 earthquakes were removed. So apart from the mainshock, no sequence contains an M ≥ 6 event.

The downloaded folder contains `sequences_summary.csv` (one row per mainshock), `jma_sequences_mirwald2026.csv.gz` (all events; `distance_to_main` is the distance to the mainshock hypocenter in km) and `japan_coastline.csv`.

In [ ]:
summary = pd.read_csv(DATA_DIR / "sequences_summary.csv", parse_dates=["time"])
events = pd.read_csv(DATA_DIR / "jma_sequences_mirwald2026.csv.gz", parse_dates=["time"])
coast = pd.read_csv(DATA_DIR / "japan_coastline.csv")

print(summary)
print(events)
print(f"{len(summary)} sequences, {len(events):,} events")

def get_sequence(seq_id):
    '''Returns the mainshock (pandas Series) and all other events of a sequence (SeismoStats Catalog).'''
    ev = events[events.sequence_id == seq_id]
    main = ev[ev.is_main_event].iloc[0]
    cat = Catalog(ev[~ev.is_main_event].drop(columns=["sequence_id", "is_main_event"])
                  .sort_values("time").reset_index(drop=True))
    return main, cat


fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(coast.lon, coast.lat, color="k", lw=0.5)
ax.scatter(events.longitude, events.latitude, s=0.05, color="0.7", rasterized=True)
sc = ax.scatter(summary.longitude, summary.latitude, s=12 * 3 ** (summary.magnitude - 6), c=summary.magnitude,
                cmap="plasma", edgecolor="k", linewidth=0.5, zorder=3)
for sid, name in {19: "2008 Iwate–Miyagi M7.2", 56: "2024 Noto Peninsula M7.6"}.items():
    row = summary.set_index("sequence_id").loc[sid]
    ax.annotate(name, (row.longitude, row.latitude), xytext=(-150, 25), textcoords="offset points", fontsize=9,
                bbox=dict(boxstyle="round", fc="white", ec="0.6"), arrowprops=dict(arrowstyle="-", color="0.3"))
plt.colorbar(sc, ax=ax, shrink=0.6, label="mainshock magnitude")
ax.set_xlim(128, 147.5)
ax.set_ylim(30, 46)
ax.set_aspect(1 / np.cos(np.radians(38)))
ax.set_title("62 mainshocks (circles) and their sequences (grey)")
plt.show()

In [ ]:

print(summary.head(n=20))
print(events.head(n=20))
print(events.loc[events.sequence_id == 1].magnitude.max())

## Part 1 — The Gutenberg–Richter law: $M_c$, b and a

### 1.1 Our catalog

We start with the seismicity **before** the 2008 Iwate–Miyagi Nairiku earthquake (M7.2, sequence 19): all events from January 2000 until the mainshock within $2L \approx 110$ km of its hypocenter. For now, this is just a regional earthquake catalog.

In [ ]:
main, cat = get_sequence(19)
before = cat[cat.time < main.time]    # the catalog we analyze in Part 1
after = cat[cat.time > main.time]     # the aftershocks (day 1 to 100), used in Section 1.6
mags = before.magnitude.values
print(main)
print(before)

print(f"Mainshock: {main.time:%Y-%m-%d}, M{main.magnitude}")
print(f"Catalog before the mainshock: {len(before)} events, {before.time.min():%Y-%m-%d} to {before.time.max():%Y-%m-%d}, "
      f"magnitudes {mags.min()} to {mags.max()}")

ax = before.plot_mags_in_time(dot_smallest=1, dot_largest=150, color_dots="tab:blue")
ax.set_title("Magnitudes in time (before the mainshock)")
plt.show()

### 1.2 The frequency–magnitude distribution (FMD)

The **frequency–magnitude distribution** counts how many earthquakes there are of each size:

* the *non-cumulative* FMD counts the events in each magnitude bin,
* the *cumulative* FMD counts the events with magnitude **≥ M**.

Gutenberg & Richter (1944) found that, above some magnitude, the cumulative FMD is a straight line on a logarithmic axis:

$$\log_{10} N(\ge M) = a - b\,M .$$

* The **b-value** is the slope. It describes how many small events there are per large event. With $b = 1$, there are 10 times more events with M ≥ 3 than with M ≥ 4.
* The **a-value** describes the overall level of seismicity, i.e. how productive the region is ($\log_{10}$ of the number of events with M ≥ 0).
* Below the **magnitude of completeness $M_c$**, the line bends down. Small earthquakes are not missing in nature, but the seismic network does not detect all of them.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.5))
plot_fmd(mags, fmd_bin=DELTA_M, ax=axs[0], color="tab:blue", label="non-cumulative")
plot_cum_fmd(mags, fmd_bin=DELTA_M, ax=axs[1], color="tab:blue", label="cumulative")
axs[0].set_title("Non-cumulative FMD")
axs[1].set_title("Cumulative FMD")
plt.show()

#### Exercise 1

Where does the FMD stop being a straight line? Estimate $M_c$ and the slope b by eye.

### 1.3 Magnitude of completeness $M_c$

$M_c$ is the lowest magnitude above which (practically) all earthquakes in the region are recorded. Every b- and a-value estimate uses **only events with $M \ge M_c$**. With $M_c$ too low, we include an incomplete part of the FMD and b is underestimated. With $M_c$ too high, we throw away data. Two common methods:

* **Maximum curvature (MAXC)** (Wiemer & Wyss, 2000): $M_c$ is the magnitude bin with the most events in the non-cumulative FMD. This tends to underestimate $M_c$, so a correction of **+0.2** is usually added (Woessner & Wiemer, 2005).
* **Kolmogorov–Smirnov (KS) test** (e.g. Mizrahi et al., 2021): starting from low values, each candidate $M_c$ is tested to see whether the magnitudes above it are consistent with a GR distribution. $M_c$ is the lowest value that passes the test (p-value ≥ 0.1).

A third, visual check is to plot the b-value estimated with different cut-off magnitudes. Below $M_c$ the estimate increases with the cut-off magnitude. Above $M_c$ it becomes stable, apart from random fluctuations that grow because fewer events are left.

In [ ]:
mc_maxc, _ = estimate_mc_maxc(mags, fmd_bin=0.1, correction_factor=0.2)
mc_ks, ks_info = estimate_mc_ks(mags, delta_m=DELTA_M, n=2000)
print(f"Mc (maximum curvature + 0.2) = {mc_maxc:.1f}")
print(f"Mc (KS test)                 = {mc_ks:.1f}   (tested Mc values: {np.round(ks_info['mcs_tested'], 1)}, "
      f"p-values: {np.round(ks_info['p_values'], 2)})")

ax = plot_mc_vs_b(mags, mcs=np.arange(0.5, 2.6, 0.1), delta_m=DELTA_M)
ax.axvline(mc_maxc, color="k", ls="--", label="Mc (MAXC + 0.2)")
ax.legend()
ax.set_title("b-value as a function of the cut-off magnitude")
plt.show()

mc = mc_maxc

### 1.4 The b-value

The b-value is estimated with the **maximum-likelihood estimator** of Aki (1965). For magnitudes given in bins of width $\Delta M$ we must use $M_c - \Delta M/2$ as the lower limit (Utsu, 1966):

$$\hat b = \frac{\log_{10} e}{\overline{M} - \left(M_c - \Delta M/2\right)}, \qquad \sigma_{\hat b} \approx \frac{\hat b}{\sqrt{n}} ,$$

where $\overline{M}$ is the mean magnitude of the $n$ events with $M \ge M_c$. Note that it only needs the *mean* magnitude: the larger the mean magnitude, the flatter the FMD and the lower b. Do **not** fit a straight line to the FMD with least squares. The bins with few large events would get far too much weight.

SeismoStats implements the exact version for binned magnitudes (Tinti & Mulargia, 1987) and the uncertainty of Shi & Bolt (1982). Let's compare both.

In [ ]:
m_complete = mags[mags >= mc - DELTA_M / 2]            # events above Mc (careful with the binning!)
print(len(m_complete))
print(len(mags))
print(100*len(m_complete)/len(mags))
n = len(m_complete)
#n = 50
b_aki = np.log10(np.e) / (m_complete.mean() - (mc - DELTA_M / 2))
print(f"By hand (Aki-Utsu): b = {b_aki:.3f} ± {b_aki / np.sqrt(n):.3f}   (n = {n} events with M >= {mc:.1f})")

estimator = ClassicBValueEstimator()
b_value = estimator.calculate(mags, mc=mc, delta_m=DELTA_M)
###
a_value = estimate_a(m_complete,mc=mc,delta_m=0.1)
print(a_value)
####
print(f"SeismoStats:        b = {b_value:.3f} ± {estimator.std:.3f}   (Shi & Bolt uncertainty)")

ax = plot_cum_fmd(mags, mc=mc, b_value=b_value, fmd_bin=DELTA_M, color="tab:blue", label="observed")
ax.axvline(mc, color="grey", ls="--")
plt.show()

#### Exercise 2

* How many events would you need to know b to within ±0.1? And within ±0.02?
* Why can't we estimate b reliably from, say, 20 earthquakes?

### 1.5 The a-value and a first (time-independent) forecast

**From counts to an annual a-value.** The GR law describes the number of earthquakes in our catalog, which covers $T$ years:

$$N(\ge m) = 10^{\,a - b\,m} .$$

We know b (Section 1.4), and we can count the events above $M_c$. Writing the law for $m = M_c$ and taking the logarithm gives

$$a = \log_{10} N(\ge M_c) + b\,M_c .$$

This a-value depends on how long we recorded: a catalog twice as long contains twice as many events, and its a-value is larger by $\log_{10} 2 \approx 0.3$. To compare regions or to forecast, we need a *rate*. Dividing by $T$ gives

$$\frac{N(\ge m)}{T} = 10^{\,a - \log_{10} T - b\,m} = 10^{\,a_T - b\,m}, \qquad a_T = a - \log_{10} T = \log_{10}\frac{N(\ge M_c)}{T} + b\,M_c .$$

$a_T$ is the **annual a-value**: $10^{a_T}$ would be the number of events with $M \ge 0$ per year if the GR law held down to $M = 0$. In the code below, `estimate_a(..., scaling_factor=T, m_ref=0)` computes the same quantity: `m_ref=0` refers the a-value to $M \ge 0$, and `scaling_factor=T` divides by $T$.

**The annual rate $\lambda(\ge m)$.** The left-hand side, $N(\ge m)/T$, is the average number of events with $M \ge m$ per year. We call it $\lambda(\ge m)$. Inserting $a_T$ shows where it comes from:

$$\lambda(\ge m) = 10^{\,a_T - b\,m} = \underbrace{\frac{N(\ge M_c)}{T}}_{\text{observed rate above } M_c} \cdot \underbrace{10^{-b\,(m - M_c)}}_{\text{fraction of these with } M \ge m} .$$

The first factor is simply counted. The second factor is the GR law: each additional magnitude unit divides the number of events by $10^{b}$ (about 6 for our b-value, 10 for $b = 1$). For magnitudes far above the largest one in the catalog (here M 4.8), this is an extrapolation of the straight line in the FMD.

**Return period.** If on average $\lambda$ events occur per year, the mean time between two of them is

$$T_r(m) = \frac{1}{\lambda(\ge m)} .$$

This is an average, not a schedule: the times between events are random, so the next one may come much earlier or much later.

**Probability of at least one event in the next $t$ years.** We assume that earthquakes occur independently of each other and at a constant rate (a **Poisson process**). Split the $t$ years into $n$ short intervals. In each interval, an event occurs with the small probability $\lambda t / n$, independently of the other intervals. The probability that no interval contains an event is then

$$P(\text{no event}) = \lim_{n \to \infty} \left(1 - \frac{\lambda t}{n}\right)^{n} = e^{-\lambda t}$$

(this is the $k = 0$ term of the Poisson distribution $P(k) = (\lambda t)^k e^{-\lambda t} / k!$). The probability of at least one event with $M \ge m$ in $t$ years is therefore

$$P = 1 - e^{-\lambda(\ge m)\, t} \;\approx\; \lambda t \quad \text{if } \lambda t \ll 1 .$$

Waiting one return period ($\lambda t = 1$) gives $P = 1 - 1/e \approx 0.63$, not 1. Because $\lambda$ is constant, the probability is the same every year, no matter when the last event occurred: the forecast is **time-independent**.

**How much does b matter?** Without a local estimate, forecasts often assume the generic value $b = 1$. We compare it with our local b-value. Both GR lines must reproduce the observed rate above $M_c$ (the first factor above), so changing b rotates the line around the point $\big(M_c,\, N(\ge M_c)/T\big)$, and the a-value changes with it.

In [ ]:
# Worked example: forecast for M >= 7 in the next t = 7,10 years, step by step
m_target, t = 6, 10
n_mc = n
T_years = 7
rate_mc = n_mc / T_years                          # observed events per year with M >= Mc
fraction = 10 ** (-b_value * (m_target - mc))     # GR: fraction of these events with M >= m_target
rate = rate_mc * fraction                         # lambda(>= m_target), in events per year
p_none = np.exp(-rate * t)                        # Poisson: probability of no event in t years
###
a_total = a_value
a_annual = a_total - np.log10(T_years)
###
print(f"\nWorked example for M >= {m_target} in the next {t} years:")
print(f"  annual a-value a_T     = a - log10(T) = {a_total:.3f} - {np.log10(T_years):.3f} = {a_annual:.3f}")
print(f"  rate above Mc          = N(M >= {mc:.1f}) / T = {n_mc} / {T_years:.2f} = {rate_mc:.0f} events/year")
print(f"  fraction with M >= {m_target}   = 10^(-b (m - Mc)) = 10^(-{b_value:.3f} x {m_target - mc:.1f}) = {fraction:.3g}")
print(f"  lambda(>= {m_target})           = {rate_mc:.0f} x {fraction:.3g} = {rate:.4f} events/year")
print(f"                         = 10^(a_T - b m) = 10^({a_annual:.3f} - {b_value:.3f} x {m_target}) = "
      f"{10 ** (a_annual - b_value * m_target):.4f} events/year")
print(f"  return period          = 1 / lambda = {1 / rate:.0f} years")
print(f"  P(no event)            = exp(-lambda t) = exp(-{rate * t:.4f}) = {p_none:.3f}")
print(f"  P(at least one event)  = 1 - exp(-lambda t) = {1 - p_none:.1%}")

#### Exercise 3

* The M7.2 Iwate–Miyagi earthquake happened in June 2008. Was it "unexpected" according to these forecasts?
* By what factor do the return periods of an M ≥ 7 differ between the local b-value and $b = 1$? Why does the difference grow with magnitude?
* Which assumptions does this forecast make? Is the rate of earthquakes constant (plot in 1.1)? And can an M8 occur in a region with a radius of 110 km? (Hint: rupture length.)

### 1.6 After a large earthquake: incompleteness and b-positive

Right after a large earthquake, many small aftershocks are hidden in the seismic signals of larger ones. The catalog is temporarily much less complete. This is called *short-term aftershock incompleteness* (STAI). The cumulative FMD of the aftershocks bends down already at magnitudes well above the $M_c$ of the normal catalog. With the standard method, b then comes out too low.

The **b-positive** estimator (van der Elst, 2021) avoids this. It does not use the magnitudes themselves but the **differences between consecutive magnitudes**, keeping only the positive ones ($m_i - m_{i-1} \ge 0.1$). These differences follow an exponential distribution with the same b-value. And an event that is larger than the previous (detected) one is almost always detected itself, even during STAI.

In [ ]:
ax = plot_cum_fmd(before.magnitude, fmd_bin=DELTA_M, color="tab:blue", label="before mainshock (2000–2008)")
plot_cum_fmd(after.magnitude, fmd_bin=DELTA_M, ax=ax, color="tab:red", label="aftershocks (day 1–100)")
ax.axvline(mc, color="grey", ls="--")
plt.show()

for name, c in [("before", before), ("after ", after)]:
    classic = ClassicBValueEstimator()
    classic.calculate(c.magnitude, mc=mc, delta_m=DELTA_M)
    positive = BPositiveBValueEstimator()
    positive.calculate(c.magnitude, mc=mc, delta_m=DELTA_M, times=c.time, dmc=0.1)
    print(f"{name}: classic b = {classic.b_value:.2f} ± {classic.std:.2f},   b-positive = {positive.b_value:.2f} ± {positive.std:.2f}")

#### Exercise 4

For which of the two periods do the classic estimator and b-positive disagree, and why?

## Part 2 — Can b-values help us forecast?

Section 1.5 showed that the forecast probability of large events depends strongly on b. A forecast extrapolates from small to large magnitudes, and the extrapolation goes with $10^{-b\,\Delta M}$. Let's apply this to a real case, and then check whether local b-values are stable enough to be used in forecasts.

### 2.1 New Year's Eve 2023 on the Noto Peninsula

Imagine it is 31 December 2023. Since late 2020, an intense earthquake swarm has been active below the tip of the Noto Peninsula, and on 5 May 2023 it produced an M6.5 earthquake. **What is the probability of an M ≥ 7 earthquake within 50 km during 2024?**

We collect all events from 2014 to 2023 within 50 km (epicentral distance) of what will be the epicenter of the 2024 mainshock. The center of the swarm lies within 37 km of the M6.5 of May 2023. Its events are therefore stored in sequence 53, because the selection removed them from sequence 56 (Part 0). We combine both sequences.

In [ ]:
def epicentral_distance_km(lat1, lon1, lat2, lon2):
    '''Great-circle distance in km (haversine formula).'''
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    h = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371.0 * np.arcsin(np.sqrt(h))

events53 = events.loc[events.sequence_id == 53]
events56 = events.loc[events.sequence_id == 56]
print(events53,events56)
noto_main = events[(events.sequence_id == 56) & events.is_main_event].iloc[0]
noto = events[events.sequence_id.isin([53, 56])
              & (events.time >= pd.Timestamp("2014-01-01")) & (events.time < pd.Timestamp("2024-01-01"))]
noto = noto[epicentral_distance_km(noto_main.latitude, noto_main.longitude, noto.latitude, noto.longitude) <= 50]
noto = Catalog(noto.sort_values("time").reset_index(drop=True))
print(noto)
mags_n, times_n = noto.magnitude.values, noto.time.values
print(f"{len(noto)} events within 50 km from 2014 to 2023; largest magnitude: {mags_n.max()}")

ax = noto.plot_mags_in_time(dot_smallest=1, dot_largest=150, color_dots="tab:blue")
ax.set_title("Magnitudes in time (within 50 km)")
plt.show()

Now we estimate $M_c$ and b exactly as in Part 1: $M_c$ with MAXC + 0.2 and with the KS test, and b with the classic estimator and with b-positive.

In [ ]:
mc_n_maxc, _ = estimate_mc_maxc(mags_n, fmd_bin=0.1, correction_factor=0.2)
mc_n_ks, _ = estimate_mc_ks(mags_n, delta_m=DELTA_M, n=1000)
print(f"Mc: MAXC + 0.2 = {mc_n_maxc:.1f},   KS test = {mc_n_ks:.1f}\n")

b_classic_n = {}
for mc_x in [mc_n_maxc, mc_n_ks]:
    classic = ClassicBValueEstimator()
    classic.calculate(mags_n, mc=mc_x, delta_m=DELTA_M)
    positive = BPositiveBValueEstimator()
    positive.calculate(mags_n, mc=mc_x, delta_m=DELTA_M, times=times_n, dmc=0.1)
    b_classic_n[mc_x] = classic.b_value
    print(f"Above Mc = {mc_x:.1f}:  classic b = {classic.b_value:.2f} ± {classic.std:.2f} (n = {classic.n}),   "
          f"b-positive = {positive.b_value:.2f} ± {positive.std:.2f} (n = {positive.n} differences)")

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
plot_cum_fmd(mags_n, fmd_bin=DELTA_M, ax=axs[0], color="tab:blue", label="observed (2014–2023)")
m_axis = np.arange(0.5, 7.51, 0.1)
for mc_x, ls in [(mc_n_maxc, "--"), (mc_n_ks, "-")]:
    n_x = (mags_n >= mc_x - DELTA_M / 2).sum()
    m_line = m_axis[m_axis >= mc_x - 1e-6]
    axs[0].plot(m_line, n_x * 10 ** (-b_classic_n[mc_x] * (m_line - mc_x)), color="k", ls=ls,
                label=f"GR above Mc = {mc_x:.1f} (b = {b_classic_n[mc_x]:.2f})")
axs[0].axvline(7, color="grey", ls=":")
axs[0].set_ylim(1e-3, None)
axs[0].legend()
axs[0].set_title("Cumulative FMD, extrapolated to M7")
mcs = np.arange(0.5, 3.01, 0.1)
plot_mc_vs_b(mags_n, mcs=mcs, delta_m=DELTA_M, ax=axs[1], color="tab:blue", label="classic")
plot_mc_vs_b(mags_n, mcs=mcs, delta_m=DELTA_M, ax=axs[1], color="tab:orange", label="b-positive",
             b_method=BPositiveBValueEstimator, times=times_n, dmc=0.1)
axs[1].axvline(mc_n_maxc, color="k", ls="--", label="Mc (MAXC + 0.2)")
axs[1].axvline(mc_n_ks, color="k", ls="-", label="Mc (KS test)")
axs[1].legend()
axs[1].set_title("b-value as a function of the cut-off magnitude")
plt.show()

**Which b-value?** The FMD (left) is not one straight line: it is steep between M1 and M2 and flatter above M2. The estimated b-value therefore depends on the cut-off magnitude (right).

* **Above Mc = 0.8 (MAXC + 0.2)** the two estimators disagree (classic ≈ 0.98, b-positive ≈ 1.15). At these small magnitudes the catalog is not complete all the time, for example during swarm bursts and after the M6.5 (STAI). This biases the classic estimator low. b-positive is robust against it, as in Section 1.6.
* **Above Mc = 2.1 (KS test)** the two estimators agree (b ≈ 0.8). At these magnitudes the catalog is complete, so incompleteness no longer matters.
* **b-positive corrects for incompleteness, but not for a curved FMD.** Applied to all events, it measures the slope where most of the events are, i.e. at small magnitudes.

For the forecast of an M7, what matters is the slope of the upper part of the FMD, where the KS test accepts the GR law. We therefore use b above Mc = 2.1, from the classic estimator (b-positive gives the same).

The forecast also depends on which period we use to estimate the rate. We compare the years before the swarm, the full 10 years and the last 12 months.

In [ ]:
mc_n = mc_n_ks
b_n = b_classic_n[mc_n]
print(f"Local b-value above Mc = {mc_n:.1f}: b = {b_n:.2f}")

windows = {"2014–2019 (before the swarm)": ("2014-01-01", "2020-01-01"),
           "2014–2023 (10 years)": ("2014-01-01", "2024-01-01"),
           "2023 (last 12 months)": ("2023-01-01", "2024-01-01")}
rows = []
for name, (t1, t2) in windows.items():
    t1, t2 = pd.Timestamp(t1), pd.Timestamp(t2)
    n_window = ((noto.time >= t1) & (noto.time < t2) & (noto.magnitude >= mc_n - DELTA_M / 2)).sum()
    rate_mc = n_window / ((t2 - t1).days / 365.25)            # events per year with M >= Mc
    row = {"rate estimated from": name, f"events/year with M ≥ {mc_n:.1f}": round(rate_mc, 1)}
    for label, b in [(f"local b = {b_n:.2f}", b_n), ("b = 1", 1.0)]:
        rate7 = rate_mc * 10 ** (-b * (6 - mc_n))               # events per year with M >= 7
        row[f"P(M ≥ 7 in 2024), {label}"] = f"{1 - np.exp(-rate7):.2%}"
    rows.append(row)
pd.DataFrame(rows).set_index("rate estimated from")

On **1 January 2024 at 16:10 local time**, an M7.6 earthquake struck the Noto Peninsula. It caused severe damage and a tsunami.

#### Exercise 5

* Look at the FMD: which of the two GR lines describes the largest events better? How different are their extrapolations at M7?
* How much does the forecast change if we use $b = 1$ instead of the local b-value? And how much does it change with the period used for the rate? What would it be with the b-positive value of all events ($b \approx 1.15$)?
* Which of these forecasts would you have communicated on 31 December 2023? Does a probability of a few percent mean that the forecast failed?

## Part 3 — Summary

* **$M_c$, b and a** describe the frequency–magnitude distribution. $M_c$ must be estimated first (MAXC + 0.2, KS test, stability of b). b follows from the mean magnitude above $M_c$ (Aki, 1965), and its uncertainty is roughly $b/\sqrt{n}$. The a-value, normalized per year, turns the GR law into expected numbers, return periods and probabilities.
* After a large earthquake the catalog is temporarily incomplete (STAI). Classic b-values are then biased low, and **b-positive** avoids this.
* Forecasts of large earthquakes depend strongly on **b** and on the **rate**. For the Noto Peninsula on 31 December 2023, the probability of an M ≥ 7 in 2024 changed by about a factor of 8 between the local b-value and $b = 1$. It changed by about a factor of 50 depending on whether the rate was taken from the years before the swarm or from 2023.

## References

* Aki, K. (1965). Maximum likelihood estimate of b in the formula log N = a − bM and its confidence limits. *Bull. Earthq. Res. Inst.*, 43, 237–239.
* Gutenberg, B., & Richter, C. F. (1944). Frequency of earthquakes in California. *BSSA*, 34(4), 185–188.
* Mirwald, A., Mizrahi, L., Enescu, B., & Wiemer, S. (2026). b-Values of large earthquake sequences depend on their mainshock location. *GRL*, 53, e2025GL121450.
* Mizrahi, L., Nandan, S., & Wiemer, S. (2021). Embracing data incompleteness for better earthquake forecasting. *JGR Solid Earth*, 126, e2021JB022379.
* Reasenberg, P. A., & Jones, L. M. (1989). Earthquake hazard after a mainshock in California. *Science*, 243, 1173–1176.
* Shi, Y., & Bolt, B. A. (1982). The standard error of the magnitude-frequency b value. *BSSA*, 72(5), 1677–1687.
* Tinti, S., & Mulargia, F. (1987). Confidence intervals of b values for grouped magnitudes. *BSSA*, 77(6), 2125–2134.
* Utsu, T. (1961). A statistical study on the occurrence of aftershocks. *Geophys. Mag.*, 30, 521–605.
* van der Elst, N. J. (2021). B-positive: A robust estimator of aftershock magnitude distribution in transiently incomplete catalogs. *JGR Solid Earth*, 126, e2020JB021027.
* Wiemer, S., & Wyss, M. (2000). Minimum magnitude of completeness in earthquake catalogs. *BSSA*, 90(4), 859–869.
* Woessner, J., & Wiemer, S. (2005). Assessing the quality of earthquake catalogues: Estimating the magnitude of completeness and its uncertainty. *BSSA*, 95(2), 684–698.

Data: JMA earthquake catalog (https://www.data.jma.go.jp/eqev/data/bulletin/hypo_e.html), sequences as extracted by Mirwald (2026), https://doi.org/10.5281/zenodo.20840787.